This code exports Climate features for the ground samples<br>
Data: https://developers.google.com/earth-engine/datasets/catalog/IDAHO_EPSCOR_TERRACLIMATE<br>
Author: Nafiseh Kakhani, University of Tuebingen<br>
Date: 04/04/2023<br>
Project: Multi-modal deep learning for SOC estimation 

In [1]:
import ee
import geemap

In [2]:
ee.Authenticate()


Successfully saved authorization token.


In [3]:
ee.Initialize()

In [6]:
import os

# -----------------------
# 基础配置
# -----------------------
region = ee.FeatureCollection("projects/ee-gh05210621/assets/CN-SOC-3500")

# ✅ 动态气候变量（加入 soil / ro / def）
Climate_bands = [
    'AET', 'PET', 'pr', 'Precip_extreme', 'pdsi',
    'srad', 'tmmn', 'tmmx', 'LST_warm', 'vap', 'vpd', 'vs', 'swe',
    'soil', 'ro', 'def'
]

# ✅ 静态变量（去掉 DEM / slope，保留 SoilGrids + TWI + TPI + LULC）
Static_bands = [
    'bulk_density', 'cec', 'sand', 'silt', 'clay', 'twi', 'tpi', 'lulc'
]

# 时间索引（固定 2011–2015）
start_year = 2011
end_year = 2015
fixed_dates = [
    f"{year}{month:02d}01"
    for year in range(start_year, end_year + 1)
    for month in range(1, 13)
]

startDate = ee.Date('2011-01-01')
endDate = ee.Date('2015-12-31')
chunk = 1000
output_dir = "./climate_exports_indexed_2011_2015"
os.makedirs(output_dir, exist_ok=True)
pointsList = region.toList(region.size())
end = pointsList.size().getInfo()


# -----------------------
# 功能函数
# -----------------------
def map_climate(feature, Climatefeature, bandName):
    def map_fixed_date(date_str):
        ini = ee.Date.parse("YYYYMMdd", date_str)
        end = ini.advance(1, 'month')
        data = Climatefeature.filterDate(ini, end).mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feature.geometry(),
            scale=1000
        )
        val = ee.Number(data.get(bandName))
        return ee.Number(ee.Algorithms.If(val, val, -999))

    values = ee.List(fixed_dates).map(map_fixed_date)
    time_dict = ee.Dictionary.fromLists(fixed_dates, values)
    return feature.set(time_dict).setGeometry(None)


def map_static(feature, static_img):
    data = static_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry(),
        scale=250
    )
    return feature.set(data).setGeometry(None)


def export_chunk(var, i, source_img, bandName, mode="climate"):
    filename = f"{var}_{i}_CN_SOC_2011_2015.csv"
    file_path = os.path.join(output_dir, filename)

    if os.path.exists(file_path):
        print(f"✅ Skipping existing: {filename}")
        return True

    try:
        if mode == "climate":
            subset = ee.FeatureCollection(region.toList(chunk, i)).map(
                lambda f: map_climate(f, source_img, bandName)
            )
        else:
            subset = ee.FeatureCollection(region.toList(chunk, i)).map(
                lambda f: map_static(f, source_img)
            )

        print(f"⏳ Exporting {filename} ...")
        geemap.ee_to_csv(subset, filename=file_path)
        print(f"✅ Done: {filename}")
        return True

    except Exception as e:
        print(f"❌ Failed: {filename}: {e}")
        with open(file_path + ".failed", "w") as f:
            f.write(str(e))
        return False


# -----------------------
# 主循环
# -----------------------

# 1️⃣ 动态气候变量
for var in Climate_bands:
    print(f"\n🔹 Processing dynamic variable: {var}")

    terraclimate_vars = [
        'AET', 'PET', 'pr', 'pdsi', 'srad',
        'tmmn', 'tmmx', 'vap', 'vpd', 'vs', 'swe',
        'soil', 'ro', 'def'
    ]

    # TerraClimate 变量
    if var in terraclimate_vars:
        Climatefeature = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').select(var.lower())
        bandName = var.lower()

    # 极端降水（ERA5-Land）
    elif var == 'Precip_extreme':
        Climatefeature = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').select('total_precipitation')
        bandName = 'total_precipitation'

    # MODIS LST
    elif var == 'LST_warm':
        Climatefeature = ee.ImageCollection('MODIS/061/MOD11A2').select('LST_Day_1km')
        bandName = 'LST_Day_1km'

    else:
        continue

    for i in range(0, end, chunk):
        export_chunk(var, i, Climatefeature, bandName, mode="climate")


# -----------------------
# 2️⃣ 静态变量
# -----------------------
print("\n🏔 Processing static layers...")

use_openlandmap = False
soilgrids = None

try:
    soilgrids = ee.Image('projects/soilgrids-isric/soilgrids_250m')
    print('SoilGrids band names (head):', soilgrids.bandNames().getInfo()[:20])
except Exception as e:
    print('⚠️ SoilGrids not accessible, fallback to OpenLandMap. Reason:', str(e)[:120])
    use_openlandmap = True

try:
    print('WorldCover band names:', ee.Image('ESA/WorldCover/v200/2021').bandNames().getInfo())
except Exception as e:
    print('⚠️ WorldCover band query failed:', str(e)[:120])

if not use_openlandmap:
    static_sources = {
        'bulk_density': soilgrids.select('bdod_0-5cm_mean'),
        'cec': soilgrids.select('cec_0-5cm_mean'),
        'sand': soilgrids.select('sand_0-5cm_mean'),
        'silt': soilgrids.select('silt_0-5cm_mean'),
        'clay': soilgrids.select('clay_0-5cm_mean'),
        'lulc': ee.Image('ESA/WorldCover/v200/2021').select('Map')
    }
else:
    static_sources = {
        'bulk_density': ee.Image('OpenLandMap/SOL/SOL_BULK-DENSITY_USDA-4A1A2A_M/v02').select('b0'),
        'cec': ee.Image('OpenLandMap/SOL/SOL_CEC-CLAY_USDA-4B1C_M/v02').select('b0'),
        'sand': ee.Image('OpenLandMap/SOL/SOL_TEXTURE-SAND-USDA-TT_M/v02').select('b0'),
        'silt': ee.Image('OpenLandMap/SOL/SOL_TEXTURE-SILT-USDA-TT_M/v02').select('b0'),
        'clay': ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLAY-USDA-TT_M/v02').select('b0'),
        'lulc': ee.Image('ESA/WorldCover/v200/2021').select('Map')
    }

# TWI / TPI（示意）
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem)
tpi = ee.Terrain.hillshade(dem)
twi = dem.unitScale(0, 3000).multiply(slope.unitScale(0, 60))

static_sources['twi'] = twi
static_sources['tpi'] = tpi

for var, img in static_sources.items():
    img = img.updateMask(img.mask())
    print(f"Exporting static layer: {var}")

    for i in range(0, end, chunk):
        export_chunk(var, i, img, bandName=var, mode="static")


# -----------------------
# 自动重试机制
# -----------------------
failed_files = [f for f in os.listdir(output_dir) if f.endswith('.failed')]
if failed_files:
    print(f"\n♻️ Retrying {len(failed_files)} failed chunks...\n")

for ff in failed_files:
    target = ff.replace('.failed', '')
    var = target.split("_")[0]
    i = int(target.split("_")[1])
    os.remove(os.path.join(output_dir, ff))
    print(f"🔁 Retrying {target}")

    if var in Climate_bands:
        mode = "climate"
        if var.lower() in terraclimate_vars:
            src = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').select(var.lower())
            band = var.lower()
        elif var == 'Precip_extreme':
            src = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').select('total_precipitation')
            band = 'total_precipitation'
        elif var == 'LST_warm':
            src = ee.ImageCollection('MODIS/061/MOD11A2').select('LST_Day_1km')
            band = 'LST_Day_1km'
        else:
            continue
    else:
        mode = "static"
        src = static_sources[var]
        band = var

    export_chunk(var, i, src, band, mode=mode)

print("\n🎉 All dynamic + static exports completed successfully.")



🔹 Processing dynamic variable: AET
✅ Skipping existing: AET_0_CN_SOC_2011_2015.csv
✅ Skipping existing: AET_1000_CN_SOC_2011_2015.csv
✅ Skipping existing: AET_2000_CN_SOC_2011_2015.csv
✅ Skipping existing: AET_3000_CN_SOC_2011_2015.csv

🔹 Processing dynamic variable: PET
✅ Skipping existing: PET_0_CN_SOC_2011_2015.csv
✅ Skipping existing: PET_1000_CN_SOC_2011_2015.csv
✅ Skipping existing: PET_2000_CN_SOC_2011_2015.csv
✅ Skipping existing: PET_3000_CN_SOC_2011_2015.csv

🔹 Processing dynamic variable: pr
✅ Skipping existing: pr_0_CN_SOC_2011_2015.csv
✅ Skipping existing: pr_1000_CN_SOC_2011_2015.csv
✅ Skipping existing: pr_2000_CN_SOC_2011_2015.csv
✅ Skipping existing: pr_3000_CN_SOC_2011_2015.csv

🔹 Processing dynamic variable: Precip_extreme
✅ Skipping existing: Precip_extreme_0_CN_SOC_2011_2015.csv
✅ Skipping existing: Precip_extreme_1000_CN_SOC_2011_2015.csv
✅ Skipping existing: Precip_extreme_2000_CN_SOC_2011_2015.csv
✅ Skipping existing: Precip_extreme_3000_CN_SOC_2011_2015.csv

🔹